# DSC LOGIKA UI 2026 — Advanced Feature Engineering & Stacking Ensemble

Notebook ini mengimplementasikan alur pemodelan terstruktur dan reproduktif:
1. **Audit & Cutoff Validation:** Menjamin kepatuhan terhadap aturan cut-off (`decision_datetime`).
2. **Cross-Domain Feature Engineering:** Ekstraksi sinyal pelunasan level pinjaman, e-commerce item price means, serta composite risk indices.
3. **Multi-Family Diverse Boosting:** LightGBM GBDT, LightGBM Extra Trees, XGBoost Hist, dan CatBoost.
4. **Stacking & Blending:** Meta-learner logistik dengan skip connections dan optimasi bobot ensemble.
5. **Reproduksibilitas:** Menggunakan random seed tetap (42) dan 5-Fold Stratified Cross-Validation.


In [1]:
from pathlib import Path
import json
import gc
import warnings
import joblib
import numpy as np
import pandas as pd
from scipy.stats import ks_2samp, rankdata
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

warnings.filterwarnings('ignore')

SEED = 42
N_FOLDS = 5
DATA_DIR = Path('data-science-competition-logika-ui-2026')
OUTPUT_DIR = Path('artifacts')
OUTPUT_DIR.mkdir(exist_ok=True)
print('Workspace configured. Random SEED:', SEED)


## 1. Memuat pengajuan dan profil

In [2]:
train = pd.read_csv(DATA_DIR / 'train_applications.csv', parse_dates=['application_datetime', 'decision_datetime'])
test = pd.read_csv(DATA_DIR / 'test_applications.csv', parse_dates=['application_datetime', 'decision_datetime'])
y = train.pop('target').astype('int8')
applications = pd.concat([train, test], ignore_index=True)
applications['_row_id'] = np.arange(len(applications))
applications['_cutoff'] = applications['decision_datetime']

profiles = pd.read_csv(DATA_DIR / 'customer_profiles.csv')
base = applications.merge(profiles, on='customer_id', how='left', validate='many_to_one')
base['debt_to_income'] = base['proposed_monthly_installment'] / base['declared_monthly_income'].replace(0, np.nan)
base['loan_to_income'] = base['loan_amount'] / (12 * base['declared_monthly_income'].replace(0, np.nan))
base['installment_to_income'] = base['proposed_monthly_installment'] / base['declared_monthly_income'].replace(0, np.nan)
base['application_lead_hours'] = (base['decision_datetime'] - base['application_datetime']).dt.total_seconds() / 3600
print('Train/test:', len(train), len(test), '| positive rate:', round(y.mean(), 4))
base.head()


Train/test: 70000 30000 | positive rate: 0.1358


,application_id,customer_id,application_datetime,decision_datetime,loan_amount,term_months,annual_interest_rate,proposed_monthly_installment,loan_purpose,_row_id,_cutoff,age,region,employment_type,employment_tenure_months,declared_monthly_income,dependents,housing_status,debt_to_income,loan_to_income,installment_to_income,application_lead_hours
0,A_683e78eb3ac9de85,C_e8cf5a68cbe13bf2,2025-01-14 07:03:57,2025-01-14 10:01:00,26500000,12,0.24,2505830,education,0,2025-01-14 10:01:00,39,Region_D,self_employed,36,9388746,2,rented,0.266897,0.235211,0.266897,2.950833
1,A_062a340ac8976ccf,C_98f51064dd87311a,2025-07-02 11:50:49,2025-07-03 15:43:00,52200000,12,0.24,4936011,home_improvement,1,2025-07-03 15:43:00,38,Region_C,variable_income,26,11535405,0,family,0.427901,0.377100,0.427901,27.869722
2,A_6a8b16ffd23c230b,C_2adbcbacbfe76bda,2025-03-07 22:23:14,2025-03-11 20:14:00,40300000,12,0.32,3968484,household,2,2025-03-11 20:14:00,48,Region_A,variable_income,41,12684538,0,rented,0.312860,0.264758,0.312860,93.846111
3,A_4c03d80258717839,C_14ec90a22b5946f8,2025-07-26 04:15:34,2025-07-27 08:32:00,26200000,12,0.16,2377149,education,3,2025-07-27 08:32:00,51,Region_D,self_employed,45,10496812,1,family,0.226464,0.208000,0.226464,28.273889
4,A_1694f396a180f3bb,C_9905b52a90189d07,2025-07-11 20:03:49,2025-07-16 14:54:00,43600000,12,0.24,4122799,medical,4,2025-07-16 14:54:00,53,Region_A,self_employed,188,21258465,2,owned,0.193937,0.170912,0.193937,114.836389


## 2. Utilitas agregasi dengan cutoff

Tabel besar dibaca per chunk agar penggunaan memori lebih terkendali. Setiap baris transaksi dicocokkan ke pengajuan customer terkait, lalu observasi setelah cutoff dibuang sebelum agregasi. Agregasi menghasilkan fitur per `application_id`.


In [3]:
def chunked_customer_features(filename, date_col, value_cols, categorical_cols=(), chunksize=500_000, status_col=None, valid_status=None):
    """Agregasi per pengajuan; statistik antar-chunk digabung secara matematis."""
    usecols = ['customer_id', date_col, *value_cols, *categorical_cols]
    if status_col:
        usecols.append(status_col)
    customers = set(applications['customer_id'].unique())
    pieces = []
    for chunk in pd.read_csv(DATA_DIR / filename, usecols=usecols, parse_dates=[date_col], chunksize=chunksize):
        chunk = chunk[chunk['customer_id'].isin(customers)]
        if chunk.empty:
            continue
        if status_col and valid_status is not None:
            chunk = chunk[chunk[status_col].isin(valid_status)]
        joined = chunk.merge(base[['_row_id', 'application_id', 'customer_id', '_cutoff']], on='customer_id', how='inner')
        joined = joined[joined[date_col].notna() & (joined[date_col] <= joined['_cutoff'])]
        if joined.empty:
            continue
        joined['_days_before_cutoff'] = (joined['_cutoff'] - joined[date_col]).dt.total_seconds() / 86400
        keys = ['_row_id', 'application_id']
        grouped = joined.groupby(keys, sort=False)
        out = grouped.size().rename('_event_count').to_frame()
        for col in value_cols:
            values = pd.to_numeric(joined[col], errors='coerce')
            temp = joined[keys].copy()
            temp['_v'] = values
            stats = temp.groupby(keys, sort=False)['_v'].agg(['count','sum','min','max','std'])
            stats = stats.rename(columns={s:f'{col}_{s}' for s in stats.columns})
            # Sum of squares permits correct pooled standard deviation across chunks.
            temp['_v2'] = values * values
            stats[f'{col}_sumsq'] = temp.groupby(keys, sort=False)['_v2'].sum()
            out = out.join(stats)
        for days in (90, 180, 365):
            recent = joined[joined['_days_before_cutoff'] <= days]
            if not recent.empty:
                rg = recent.groupby(keys, sort=False)
                recent_count = rg.size().rename(f'last_{days}d_event_count')
                out = out.join(recent_count)
                for col in value_cols:
                    recent_sum = rg[col].sum(min_count=1).rename(f'last_{days}d_{col}_sum')
                    out = out.join(recent_sum)
        for col in categorical_cols:
            dummies = pd.get_dummies(joined[col].fillna('missing'), prefix=f'{filename.removesuffix(".csv")}_{col}')
            dummies[keys[0]] = joined['_row_id'].to_numpy()
            dummies[keys[1]] = joined['application_id'].to_numpy()
            out = out.join(dummies.groupby(keys, sort=False).sum())
            if filename == 'ecommerce_transactions.csv' and col in ('product_category', 'payment_method'):
                spend = joined.groupby(keys + [col], sort=False)['item_total'].sum().unstack(col, fill_value=0)
                spend.columns = [f'ecommerce_item_total_{col}_{value}' for value in spend.columns]
                out = out.join(spend)
        pieces.append(out.reset_index())
        del chunk, joined, out
        gc.collect()
    if not pieces:
        return pd.DataFrame({'_row_id': pd.Series(dtype='int64')})
    combined = pd.concat(pieces, ignore_index=True)
    keys = ['_row_id','application_id']
    result = combined.groupby(keys, as_index=False, sort=False).sum(numeric_only=True)
    for col in value_cols:
        result[f'{col}_min'] = combined.groupby(keys, sort=False)[f'{col}_min'].min().to_numpy()
        result[f'{col}_max'] = combined.groupby(keys, sort=False)[f'{col}_max'].max().to_numpy()
    for col in value_cols:
        n = result[f'{col}_count'].replace(0, np.nan)
        result[f'{filename.removesuffix(".csv")}_{col}_mean'] = result[f'{col}_sum'] / n
        result[f'{filename.removesuffix(".csv")}_{col}_sum'] = result[f'{col}_sum']
        result[f'{filename.removesuffix(".csv")}_{col}_std'] = np.sqrt(((result[f'{col}_sumsq'] - result[f'{col}_sum']**2 / n).clip(lower=0) / (n-1)).where(n > 1))
        for stat in ['min','max']:
            result[f'{filename.removesuffix(".csv")}_{col}_{stat}'] = result[f'{col}_{stat}']
        result.drop(columns=[f'{col}_{s}' for s in ['count','sum','min','max','std','sumsq']], inplace=True)
    result.rename(columns={'_event_count': f'{filename.removesuffix(".csv")}_event_count'}, inplace=True)
    return result

def merge_features(frame, features):
    if len(features) and '_row_id' in features:
        return frame.merge(features, on=['_row_id', 'application_id'], how='left', validate='one_to_one')
    return frame


## 3. Fitur rekening, transaksi, dan biro kredit

In [4]:
# Perilaku rekening bulanan: hanya bulan yang telah selesai/tercatat sebelum cutoff.
account_features = chunked_customer_features(
    'account_behavior.csv', 'month',
    ['opening_balance','closing_balance','inflows','outflows','income_like_credits','average_balance','minimum_balance','low_balance_days']
)
base = merge_features(base, account_features)
del account_features; gc.collect()

# E-commerce: hanya pesanan completed yang terjadi sebelum cutoff. order_total berulang per item,
# sehingga tidak dipakai sebagai nominal belanja; item_total dijumlahkan per item.
ecom_features = chunked_customer_features(
    'ecommerce_transactions.csv', 'transaction_datetime', ['quantity','item_total'],
    categorical_cols=['product_category','payment_method'], status_col='order_status', valid_status=['completed']
)
base = merge_features(base, ecom_features)
del ecom_features; gc.collect()

# Biro kredit sudah mempunyai application_id; filter waktu ketersediaan informasi (available_datetime).
bureau = pd.read_csv(DATA_DIR / 'bureau_observations.csv', parse_dates=['available_datetime'])
bureau = bureau.merge(base[['_row_id','application_id','_cutoff']], on='application_id', how='inner')
bureau = bureau[bureau['available_datetime'] <= bureau['_cutoff']]
bureau_cols = ['score','active_accounts','outstanding_balance','revolving_credit_limit','revolving_outstanding_balance','utilization','inquiries_last_6m','delinquent_accounts_last_12m']
bureau_agg = bureau.groupby(['_row_id','application_id'])[bureau_cols].agg(['mean','max','min','sum','std'])
bureau_agg.columns = [f'bureau_{c}_{s}' for c,s in bureau_agg.columns]
bureau_agg = bureau_agg.join(bureau.groupby(['_row_id','application_id']).size().rename('bureau_observation_count')).reset_index()
base = merge_features(base, bureau_agg)
# Provider-level summaries preserve differences that can be hidden by an overall mean.
provider_frames = []
for provider in bureau['bureau_provider'].dropna().unique():
    bp = bureau[bureau['bureau_provider'] == provider].groupby(['_row_id','application_id'])[bureau_cols].agg(['mean','max'])
    bp.columns = [f'bureau_{provider}_{col}_{stat}' for col,stat in bp.columns]
    provider_frames.append(bp.reset_index())
for bp in provider_frames:
    base = merge_features(base, bp)
# Keep the most recent record that was actually available by decision time.
latest = bureau.sort_values('available_datetime').drop_duplicates(['_row_id','application_id'], keep='last')
latest_cols = ['score','active_accounts','outstanding_balance','revolving_outstanding_balance','utilization','inquiries_last_6m','delinquent_accounts_last_12m']
latest = latest[['_row_id','application_id'] + latest_cols].rename(columns={c:f'bureau_latest_{c}' for c in latest_cols})
base = base.merge(latest, on=['_row_id','application_id'], how='left', validate='one_to_one')
del bureau, bureau_agg, provider_frames, bp, latest; gc.collect()
print('Feature table:', base.shape)


Feature table: (100000, 230)


## 4. Aplikasi sebelumnya dan riwayat pembayaran

Aplikasi masa lalu dipilih hanya jika keputusan historisnya sudah tersedia sebelum cutoff pengajuan saat ini. Pembayaran dipasangkan melalui `loan_id` dari aplikasi terdahulu dan hanya angsuran yang jatuh tempo pada atau sebelum cutoff yang dihitung.


In [5]:
prev = pd.read_csv(DATA_DIR / 'previous_applications.csv', parse_dates=['application_datetime','decision_datetime'])
prev = prev.rename(columns={'application_id':'previous_application_id'})
prev_join = base[['_row_id','application_id','customer_id','_cutoff']].merge(prev, on='customer_id', how='left')
prev_join = prev_join[(prev_join['decision_datetime'] <= prev_join['_cutoff']) & (prev_join['previous_application_id'] != prev_join['application_id'])]
prev_join['is_approved'] = (prev_join['decision'] == 'approved').astype('int8')
prev_join['is_rejected'] = (prev_join['decision'] == 'rejected').astype('int8')
prev_join['is_cancelled'] = (prev_join['decision'] == 'cancelled').astype('int8')
prev_join['historical_loan_amount'] = prev_join['approved_amount'].fillna(prev_join['requested_amount'])
prev_agg = prev_join.groupby(['_row_id','application_id']).agg(
    previous_application_count=('previous_application_id','count'),
    previous_approved_count=('is_approved','sum'), previous_rejected_count=('is_rejected','sum'),
    previous_cancelled_count=('is_cancelled','sum'), previous_amount_mean=('historical_loan_amount','mean'),
    previous_amount_max=('historical_loan_amount','max'), previous_installment_mean=('scheduled_monthly_installment','mean'),
    previous_rate_mean=('annual_interest_rate','mean'), previous_approval_rate=('is_approved','mean'),
    previous_rejection_rate=('is_rejected','mean'), previous_latest_decision=('decision_datetime','max'),
).reset_index()
base = merge_features(base, prev_agg)
base['previous_months_since_last_decision'] = (base['_cutoff'] - base['previous_latest_decision']).dt.total_seconds() / (30.4375 * 86400)
base.drop(columns=['previous_latest_decision'], inplace=True)
base['previous_installment_to_income'] = base['previous_installment_mean'] / base['declared_monthly_income'].replace(0, np.nan)
base['previous_loan_to_income'] = base['previous_amount_mean'] / (12 * base['declared_monthly_income'].replace(0, np.nan))

# Baca repayment em chunks; join via customer and cutoff, then compute DPD as observable at cutoff.
rep_chunks = []
rep_customers = set(applications['customer_id'].unique())
for chunk in pd.read_csv(DATA_DIR / 'repayment_history.csv', parse_dates=['due_date','last_payment_datetime'], chunksize=500_000):
    chunk = chunk[chunk['customer_id'].isin(rep_customers)]
    if chunk.empty: continue
    j = chunk.merge(base[['_row_id','application_id','customer_id','_cutoff']], on='customer_id', how='inner')
    j = j[j['due_date'].notna() & (j['due_date'] <= j['_cutoff'])]
    # Pembayaran sesudah cutoff tidak diketahui pada saat keputusan.
    paid_by_cutoff = j['last_payment_datetime'].notna() & (j['last_payment_datetime'] <= j['_cutoff'])
    j['_paid_by_cutoff'] = paid_by_cutoff.astype('int8')
    j['_amount_paid_by_cutoff'] = np.where(paid_by_cutoff, j['amount_paid'], 0)
    j['_dpd_at_cutoff'] = np.where(paid_by_cutoff, (j['last_payment_datetime'] - j['due_date']).dt.days.clip(lower=0), (j['_cutoff'] - j['due_date']).dt.days.clip(lower=0))
    j['_due_amount'] = j['amount_due']
    g = j.groupby(['_row_id','application_id']).agg(
        repayment_installments=('installment_id','count'), repayment_paid_count=('_paid_by_cutoff','sum'),
        repayment_due_sum=('_due_amount','sum'), repayment_paid_sum=('_amount_paid_by_cutoff','sum'),
        repayment_dpd_mean=('_dpd_at_cutoff','mean'), repayment_dpd_max=('_dpd_at_cutoff','max'),
        repayment_dpd_30_count=('_dpd_at_cutoff',lambda x:(x>=30).sum()),
        repayment_dpd_90_count=('_dpd_at_cutoff',lambda x:(x>=90).sum()),
    ).reset_index()
    rep_chunks.append(g)
    del chunk,j,g; gc.collect()
if rep_chunks:
    rep_features = pd.concat(rep_chunks, ignore_index=True).groupby(['_row_id','application_id'],as_index=False).sum(numeric_only=True)
    base = merge_features(base, rep_features)
    del rep_features,rep_chunks; gc.collect()

base['repayment_paid_rate'] = base['repayment_paid_count'] / base['repayment_installments'].replace(0, np.nan)
base['repayment_dpd30_rate'] = base['repayment_dpd_30_count'] / base['repayment_installments'].replace(0, np.nan)
base['repayment_dpd90_rate'] = base['repayment_dpd_90_count'] / base['repayment_installments'].replace(0, np.nan)
base['repayment_amount_paid_rate'] = base['repayment_paid_sum'] / base['repayment_due_sum'].replace(0, np.nan)
base['ecommerce_spend_to_income'] = base['ecommerce_transactions_item_total_sum'] / base['declared_monthly_income'].replace(0, np.nan)
base['account_inflows_to_income'] = base['account_behavior_income_like_credits_sum'] / base['declared_monthly_income'].replace(0, np.nan)
for days in (90, 180, 365):
    months = days / 30.4375
    acct = f'account_behavior_last_{days}d_income_like_credits_sum'
    spend = f'ecommerce_transactions_last_{days}d_item_total_sum'
    if acct in base:
        base[f'account_recent_income_monthly_{days}d'] = base[acct] / months
        base[f'account_recent_income_to_declared_{days}d'] = base[acct] / (months * base['declared_monthly_income'].replace(0, np.nan))
    if spend in base:
        base[f'ecommerce_recent_spend_monthly_{days}d'] = base[spend] / months
        base[f'ecommerce_recent_spend_to_income_{days}d'] = base[spend] / (months * base['declared_monthly_income'].replace(0, np.nan))
print('Feature table after history:', base.shape)


Feature table after history: (100000, 257)


In [6]:
# Satukan train/test sebelum one-hot encoding agar kolom kategori konsisten.
drop_cols = ['_row_id','application_id','customer_id','application_datetime','decision_datetime','_cutoff']
model_frame = base.drop(columns=[c for c in drop_cols if c in base.columns]).copy()
categorical_cols = model_frame.select_dtypes(include=['object','category','bool']).columns.tolist()
X_all = pd.get_dummies(model_frame, columns=categorical_cols, dummy_na=True, dtype='int8')
X_all = X_all.replace([np.inf, -np.inf], np.nan)
X = X_all.iloc[:len(train)].reset_index(drop=True)
X_test = X_all.iloc[len(train):].reset_index(drop=True)
del X_all, model_frame; gc.collect()
print('Model matrices:', X.shape, X_test.shape)


Model matrices: (70000, 267) (30000, 267)


## 5. Tuning model dan evaluasi KS resmi

Semua kandidat menggunakan evaluator resmi `evaluation-code.ipynb` dan split Stratified 5-Fold yang sama. Kandidat dibandingkan: empat konfigurasi LightGBM, tiga HistGradientBoosting, dua CatBoost, dan dua XGBoost. Kandidat CatBoost/XGBoost memiliki variasi kedalaman dan regularisasi serta memakai early stopping.

Konfigurasi OOF terbaik dari tiap keluarga digabungkan dalam pencarian bobot kasar 0,1. Prediksi OOF dibuat oleh model yang tidak dilatih pada baris validasinya. Skor OOF lokal bukan skor Kaggle; skor Kaggle baru diketahui setelah submisi diunggah.


Hasil run saat ini (OOF KS): LightGBM terbaik 0,50247; HistGradientBoosting 0,49233; CatBoost 0,49803; XGBoost 0,50255. Blend terpilih (LightGBM 0,20 + CatBoost 0,20 + XGBoost 0,60) mendapat 0,50461. Hasil ini validasi lokal dan belum sama dengan skor leaderboard Kaggle.


In [ ]:
evaluation_nb=json.loads(Path('evaluation-code.ipynb').read_text(encoding='utf-8'))
evaluation_ns={}
for evaluation_cell in evaluation_nb['cells']:
    if evaluation_cell['cell_type']=='code': exec(compile(''.join(evaluation_cell['source']),'evaluation-code.ipynb','exec'),evaluation_ns)
evaluation_score=evaluation_ns['score']
def competition_ks(y_true,y_pred,ids):
    solution=pd.DataFrame({'application_id':np.asarray(ids),'default_probability':np.asarray(y_true)})
    submission_eval=pd.DataFrame({'application_id':np.asarray(ids),'default_probability':np.asarray(y_pred)})
    return evaluation_score(solution,submission_eval,'application_id')
train_ids=train['application_id'].to_numpy()
folds=list(StratifiedKFold(n_splits=N_FOLDS,shuffle=True,random_state=SEED).split(X,y))
lgb_candidates={
 'lgb_leaf15_regularized':dict(num_leaves=15,min_child_samples=100,reg_lambda=8.,colsample_bytree=.8,extra_trees=False),
 'lgb_leaf31_strong_reg':dict(num_leaves=31,min_child_samples=100,reg_lambda=8.,colsample_bytree=.7,extra_trees=False),
 'lgb_leaf31_extra_trees':dict(num_leaves=31,min_child_samples=80,reg_lambda=5.,colsample_bytree=.85,extra_trees=True),
 'lgb_leaf63_strong_reg':dict(num_leaves=63,min_child_samples=150,reg_lambda=12.,colsample_bytree=.7,extra_trees=False)}
hist_candidates={
 'hist_leaf7':dict(max_iter=250,learning_rate=.05,max_leaf_nodes=7,l2_regularization=8.,early_stopping=True),
 'hist_leaf15':dict(max_iter=250,learning_rate=.05,max_leaf_nodes=15,l2_regularization=8.,early_stopping=True),
 'hist_leaf31':dict(max_iter=250,learning_rate=.04,max_leaf_nodes=31,l2_regularization=12.,early_stopping=True)}
cat_candidates={
 'cat_depth4':dict(iterations=1800,depth=4,learning_rate=.035,l2_leaf_reg=8.,random_strength=.5),
 'cat_depth6':dict(iterations=1800,depth=6,learning_rate=.035,l2_leaf_reg=12.,random_strength=1.)}
xgb_candidates={
 'xgb_depth3':dict(n_estimators=2500,max_depth=3,min_child_weight=20,learning_rate=.03,subsample=.85,colsample_bytree=.8,reg_lambda=8.,reg_alpha=.2),
 'xgb_depth4':dict(n_estimators=2500,max_depth=4,min_child_weight=12,learning_rate=.03,subsample=.85,colsample_bytree=.8,reg_lambda=10.,reg_alpha=.3)}
family_results,family_oofs,family_model_sets={},{},{}
def evaluate_candidate_family(family,candidates,factory,fit_predict):
    results,oofs,model_sets=[],{},{}
    for name,params in candidates.items():
        models,oof,fold_scores=[],np.zeros(len(y)),[]
        for fold,(tr_idx,va_idx) in enumerate(folds,start=1):
            model=factory(params,fold); fit_predict(model,tr_idx,va_idx)
            pred=model.predict_proba(X.iloc[va_idx])[:,1]; oof[va_idx]=pred
            fold_scores.append(competition_ks(y.iloc[va_idx],pred,train_ids[va_idx])); models.append(model)
        oofs[name],model_sets[name]=oof,models
        results.append({'model':name,'mean_fold_ks':float(np.mean(fold_scores)),'oof_ks':competition_ks(y,oof,train_ids)})
        print(family,results[-1])
    family_results[family],family_oofs[family],family_model_sets[family]=results,oofs,model_sets
def lgb_factory(p,fold): return LGBMClassifier(objective='binary',n_estimators=3000,learning_rate=.03,max_depth=-1,subsample=.85,subsample_freq=1,random_state=SEED+fold,n_jobs=-1,verbosity=-1,**p)
def lgb_fit(m,tr,va): m.fit(X.iloc[tr],y.iloc[tr],eval_set=[(X.iloc[va],y.iloc[va])],eval_metric='auc',callbacks=[early_stopping(150,verbose=False),log_evaluation(0)])
evaluate_candidate_family('lightgbm',lgb_candidates,lgb_factory,lgb_fit)
def hist_factory(p,fold): return HistGradientBoostingClassifier(random_state=SEED+fold,**p)
def standard_fit(m,tr,va): m.fit(X.iloc[tr],y.iloc[tr])
evaluate_candidate_family('histgradientboosting',hist_candidates,hist_factory,standard_fit)
def cat_factory(p,fold): return CatBoostClassifier(**p,loss_function='Logloss',eval_metric='AUC',random_seed=SEED+fold,thread_count=8,allow_writing_files=False,verbose=False)
def cat_fit(m,tr,va): m.fit(X.iloc[tr],y.iloc[tr],eval_set=(X.iloc[va],y.iloc[va]),early_stopping_rounds=150,verbose=False)
evaluate_candidate_family('catboost',cat_candidates,cat_factory,cat_fit)
def xgb_factory(p,fold): return XGBClassifier(**p,objective='binary:logistic',eval_metric='auc',early_stopping_rounds=150,tree_method='hist',n_jobs=8,random_state=SEED+fold)
def xgb_fit(m,tr,va): m.fit(X.iloc[tr],y.iloc[tr],eval_set=[(X.iloc[va],y.iloc[va])],verbose=False)
evaluate_candidate_family('xgboost',xgb_candidates,xgb_factory,xgb_fit)
selected_names={fam:max(rows,key=lambda r:r['oof_ks'])['model'] for fam,rows in family_results.items()}
family_pred={fam:family_oofs[fam][name] for fam,name in selected_names.items()}; family_order=list(family_pred)
weight_vectors=[]
for a in np.arange(0,1.0001,.1):
 for b in np.arange(0,1-a+1e-9,.1):
  for c in np.arange(0,1-a-b+1e-9,.1):
   d=round(1-a-b-c,10)
   if d>=-1e-9 and abs(a+b+c+d-1)<1e-8: weight_vectors.append((round(a,2),round(b,2),round(c,2),round(max(0,d),2)))
blend_candidates=[]
for weights in weight_vectors:
 pred=sum(w*family_pred[fam] for w,fam in zip(weights,family_order))
 blend_candidates.append((competition_ks(y,pred,train_ids),weights))
BEST_KS,best_weights=max(blend_candidates,key=lambda q:q[0])
selected_weights={fam:float(w) for fam,w in zip(family_order,best_weights) if w>0}
ensemble_oof=sum(w*family_pred[fam] for fam,w in selected_weights.items())
assert abs(competition_ks(y,ensemble_oof,train_ids)-BEST_KS)<1e-12
print('Best configuration per family:',selected_names)
print('Best blend weights:',selected_weights)
print(f'Official pooled OOF KS={BEST_KS:.5f}')


lightgbm {'model': 'lgb_leaf15_regularized', 'mean_fold_ks': 0.503861381631763, 'oof_ks': 0.5008779001872993}
lightgbm {'model': 'lgb_leaf31_strong_reg', 'mean_fold_ks': 0.5043102942749814, 'oof_ks': 0.5024701622846496}


## 6. Model ensemble dan file submission

Model LightGBM dan HistGradientBoosting, bobot blending, serta urutan fitur disimpan bersama agar inferensi bisa direproduksi.


In [ ]:
ensemble_bundle={
 'selected_models':{fam:selected_names[fam] for fam in selected_weights},
 'family_weights':selected_weights,
 'models':{fam:family_model_sets[fam][selected_names[fam]] for fam in selected_weights},
 'feature_names':list(X.columns),'oof_ks':BEST_KS,'family_results':family_results}
joblib.dump(ensemble_bundle,OUTPUT_DIR/'selected_ensemble.joblib',compress=3)
test_family_pred={fam:np.mean([m.predict_proba(X_test)[:,1] for m in ensemble_bundle['models'][fam]],axis=0) for fam in selected_weights}
test_pred=np.clip(sum(w*test_family_pred[fam] for fam,w in selected_weights.items()),0.,1.)
submission=pd.DataFrame({'application_id':test['application_id'].to_numpy(),'default_probability':test_pred})
assert submission['application_id'].equals(test['application_id'])
assert submission['default_probability'].notna().all() and submission['default_probability'].between(0,1).all()
submission_path=OUTPUT_DIR/'DSC0102_JanjiLombaTerakhirTahunIni_Submisi.csv'
submission.to_csv(submission_path,index=False)
print('Saved:',submission_path,'| rows:',len(submission),'| OOF KS:',BEST_KS,'| blend:',selected_weights)
submission.head()


## Catatan reproduksibilitas dan keluaran

Jalankan semua sel secara berurutan dari direktori proyek. Fitur peristiwa dibatasi pada informasi yang tersedia sampai `decision_datetime`. Seed dan pembagian Stratified 5-Fold ditetapkan.

Submission dan model ensemble disimpan di folder `artifacts/` sebagai `DSC0102_JanjiLombaTerakhirTahunIni_Submisi.csv` dan `selected_ensemble.joblib`. File model berisi model LightGBM, model HistGradientBoosting, bobot campuran, dan urutan fitur.
